In [1]:
import torch
import torch.nn as nn
import torch.onnx as onnx

In [2]:
model_path = "../simple_conv.onnx"

In [3]:
# A small CNN exercising every layer type currently supported by the Rust runtime:
# Conv2d -> ReLU -> Conv2d -> ReLU -> Flatten -> Linear -> Sigmoid -> Linear -> Tanh -> Linear -> Softmax
#
# Input is (1, 1, 4, 4): batch size 1, 1 channel, 4x4 spatial. Both convs use
# kernel_size=3, stride=1, padding=1 so spatial dims stay 4x4 throughout ("same" padding),
# which keeps the flattened size (8 * 4 * 4 = 128) easy to check by hand.


class SimpleConvModel(nn.Module):
    def __init__(self):
        super(SimpleConvModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.act_1_relu = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.act_2_relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(8 * 4 * 4, 20)
        self.act_3_sigmoid = nn.Sigmoid()
        self.linear2 = nn.Linear(20, 15)
        self.act_4_tanh = nn.Tanh()
        self.linear3 = nn.Linear(15, 5)
        self.act_5_softmax = nn.Softmax(dim=1)  # Apply softmax along the feature dimension

    def forward(self, x):
        output = self.conv1(x)
        output = self.act_1_relu(output)
        output = self.conv2(output)
        output = self.act_2_relu(output)
        output = self.flatten(output)
        output = self.linear1(output)
        output = self.act_3_sigmoid(output)
        output = self.linear2(output)
        output = self.act_4_tanh(output)
        output = self.linear3(output)
        output = self.act_5_softmax(output)
        return output


# Example usage
model = SimpleConvModel()
print(model)

print("Model weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    if "weight" in name:
        print(f"  Weight values (first 5): {param.flatten()[:5]}")
    elif "bias" in name:
        print(f"  Bias values (first 5): {param.flatten()[:5]}")

SimpleConvModel(
  (conv1): Conv2d(1, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (act_1_relu): ReLU()
  (conv2): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (act_2_relu): ReLU()
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=128, out_features=20, bias=True)
  (act_3_sigmoid): Sigmoid()
  (linear2): Linear(in_features=20, out_features=15, bias=True)
  (act_4_tanh): Tanh()
  (linear3): Linear(in_features=15, out_features=5, bias=True)
  (act_5_softmax): Softmax(dim=1)
)
Model weights:
conv1.weight: torch.Size([4, 1, 3, 3])
  Weight values (first 5): tensor([ 0.0351, -0.1801, -0.2614, -0.1696,  0.1048], grad_fn=<SliceBackward0>)
conv1.bias: torch.Size([4])
  Bias values (first 5): tensor([-0.0280,  0.0604,  0.0387,  0.3204], grad_fn=<SliceBackward0>)
conv2.weight: torch.Size([8, 4, 3, 3])
  Weight values (first 5): tensor([ 0.1355,  0.1583, -0.0546, -0.1173, -0.0658], grad_fn=<SliceBackward0>)
conv2.bias: torch.Size([8])
  

In [4]:
# export to onnx
dummy_input = torch.randn(1, 1, 4, 4)
onnx.export(model, dummy_input, model_path, export_params=True, opset_version=11)

/tmp/ipykernel_12326/3045890397.py:3: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  onnx.export(model, dummy_input, model_path, export_params=True, opset_version=11)
W0828 17:14:17.919000 12326 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleConvModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleConvModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/david/miniconda3/envs/pytorch/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/david/miniconda3/envs/pytor

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.13.0+cu132',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x"<FLOAT,[1,1,4,4]>
            ),
            outputs=(
                %"softmax"<FLOAT,[1,5]>
            ),
            initializers=(
                %"conv1.weight"<FLOAT,[4,1,3,3]>{TorchTensor(...)},
                %"conv1.bias"<FLOAT,[4]>{TorchTensor<FLOAT,[4]>(Parameter containing: tensor([-0.0280,  0.0604,  0.0387,  0.3204], requires_grad=True), name='conv1.bias')},
                %"conv2.weight"<FLOAT,[8,4,3,3]>{TorchTensor(...)},
                %"conv2.bias"<FLOAT,[8]>{TorchTensor<FLOAT,[8]>(Parameter containing: tensor([ 0.0816, -0.0424,  0.0560, -0.0140, -0.0757,  0.0124,  0.0734,  0.0063], requires_grad=True), name='conv2.bias')},
              

In [5]:
# run the model with pytorch
# 4x4 single-channel "image" with values 1..16, so it's easy to eyeball against the Rust output
input_data = torch.arange(1, 17, dtype=torch.float32).reshape(1, 1, 4, 4)
with torch.no_grad():
    output = model(input_data)
print(input_data)
print(output)

tensor([[[[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])
tensor([[0.2825, 0.1681, 0.1884, 0.1377, 0.2232]])


In [6]:
class HugeLinearModel(nn.Module):
    def __init__(self):
        super(HugeLinearModel, self).__init__()
        self.linear1 = nn.Linear(1200, 1800)
        self.linear2 = nn.Linear(1800, 1500)
        self.linear3 = nn.Linear(1500, 2000)
        self.linear4 = nn.Linear(2000, 3000)

    def forward(self, x):
        output = self.linear1(x)
        output = self.linear2(output)
        output = self.linear3(output)
        output = self.linear4(output)
        return output


# Example usage
model = HugeLinearModel()
print(model)

print("Model weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    if "weight" in name:
        print(f"  Weight values (first 5): {param.flatten()[:5]}")
    elif "bias" in name:
        print(f"  Bias values (first 5): {param.flatten()[:5]}")

# export to onnx
linear_path = "../huge_linear.onnx"
dummy_input = torch.randn(1, 1200)
onnx.export(model, dummy_input, linear_path, export_params=True, opset_version=11)

/tmp/ipykernel_12326/2260269775.py:32: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  onnx.export(model, dummy_input, linear_path, export_params=True, opset_version=11)
W0828 17:14:20.601000 12326 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


HugeLinearModel(
  (linear1): Linear(in_features=1200, out_features=1800, bias=True)
  (linear2): Linear(in_features=1800, out_features=1500, bias=True)
  (linear3): Linear(in_features=1500, out_features=2000, bias=True)
  (linear4): Linear(in_features=2000, out_features=3000, bias=True)
)
Model weights:
linear1.weight: torch.Size([1800, 1200])
  Weight values (first 5): tensor([ 0.0094, -0.0230, -0.0032, -0.0185, -0.0207], grad_fn=<SliceBackward0>)
linear1.bias: torch.Size([1800])
  Bias values (first 5): tensor([ 0.0104,  0.0043,  0.0277, -0.0041, -0.0038], grad_fn=<SliceBackward0>)
linear2.weight: torch.Size([1500, 1800])
  Weight values (first 5): tensor([-0.0044,  0.0161, -0.0030, -0.0040, -0.0124], grad_fn=<SliceBackward0>)
linear2.bias: torch.Size([1500])
  Bias values (first 5): tensor([ 0.0100,  0.0008,  0.0095,  0.0088, -0.0146], grad_fn=<SliceBackward0>)
linear3.weight: torch.Size([2000, 1500])
  Weight values (first 5): tensor([-0.0157, -0.0049,  0.0041,  0.0126,  0.0184], 

/home/david/miniconda3/envs/pytorch/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/david/miniconda3/envs/pytor

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.13.0+cu132',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x"<FLOAT,[1,1200]>
            ),
            outputs=(
                %"linear_3"<FLOAT,[1,3000]>
            ),
            initializers=(
                %"linear1.weight"<FLOAT,[1800,1200]>{TorchTensor(...)},
                %"linear1.bias"<FLOAT,[1800]>{TorchTensor(...)},
                %"linear2.weight"<FLOAT,[1500,1800]>{TorchTensor(...)},
                %"linear2.bias"<FLOAT,[1500]>{TorchTensor(...)},
                %"linear3.weight"<FLOAT,[2000,1500]>{TorchTensor(...)},
                %"linear3.bias"<FLOAT,[2000]>{TorchTensor(...)},
                %"linear4.weight"<FLOAT,[3000,2000]>{TorchTensor(...)},
                %"linear4.bias"<FLOAT,[3000]>{

In [7]:
# an rnn model in pytorch
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # The RNN layer
        # batch_first=True means input shape is (batch, seq_len, input_size)
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # A fully connected layer to map the hidden state to the desired output size
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Initialize the hidden state with zeros
        # Shape: (num_layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        # Forward propagate RNN
        # out: tensor containing the output features from the last layer of the RNN
        # hn: tensor containing the final hidden state
        out, hn = self.rnn(x, h0)

        # We usually take the output of the last time step to pass to the linear layer
        # out[:, -1, :] selects the last time step for all batches
        out = self.fc(out[:, -1, :])
        return out


rnn_model = SimpleRNN(10, 20, 2)
rnn_path = "../simple_rnn.onnx"
dummy_input = torch.randn(1, 10, 10)
onnx.export(rnn_model, dummy_input, rnn_path, export_params=True, opset_version=14)

/tmp/ipykernel_12326/1796252091.py:34: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  onnx.export(rnn_model, dummy_input, rnn_path, export_params=True, opset_version=14)
W0828 17:14:21.413000 12326 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleRNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleRNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/david/miniconda3/envs/pytorch/lib/python3.14/contextlib.py:148: UserWarning: The tensor attributes self.rnn._flat_weights[0], self.rnn._flat_weights[1], self.rnn._flat_weights[2], self.rnn._flat_weights[3] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)
/home/david/miniconda3/envs/pytorch/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 14},
            producer_name='pytorch',
            producer_version='2.13.0+cu132',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x"<FLOAT,[1,10,10]>
            ),
            outputs=(
                %"linear_11"<FLOAT,[1,2]>
            ),
            initializers=(
                %"rnn.bias_ih_l0"<FLOAT,[20]>{TorchTensor(...)},
                %"rnn.bias_hh_l0"<FLOAT,[20]>{TorchTensor(...)},
                %"fc.weight"<FLOAT,[2,20]>{TorchTensor(...)},
                %"fc.bias"<FLOAT,[2]>{TorchTensor<FLOAT,[2]>(Parameter containing: tensor([-0.0635, -0.1419], requires_grad=True), name='fc.bias')},
                %"val_11"<FLOAT,[10,20]>{Tensor(...)},
                %"linear_1"<FLOAT,[1,1,20]>{Tensor(...)},
                %"val_55"<FLOAT,[20,20]>{Tensor(...)},
                %"val